<a href="https://colab.research.google.com/github/chrishg23-jpg/HES-benchmark/blob/main/Stability005.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Stability005 — Act XXXII: The Scale Test
# Sweep over alpha and gamma to restore attractor scale

import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import fft2, fftshift
import pandas as pd
import os

# --- Parameters ---
L = 100
T_max = 100
beta = 0.01
delta = 0.02
alpha_values = [0.1, 0.3, 0.5, 0.7, 1.0]
gamma_values = [0.02, 0.05, 0.1, 0.15, 0.2]
seed = 42
save_images = False  # Set to True to save visualizations

# --- Laplacian Function ---
def laplacian(Phi):
    return (
        np.roll(Phi, 1, axis=0) + np.roll(Phi, -1, axis=0) +
        np.roll(Phi, 1, axis=1) + np.roll(Phi, -1, axis=1) -
        4 * Phi
    )

# --- Dominant Wavelength Extraction ---
def extract_lambda_dom(Phi):
    spectrum = np.abs(fftshift(fft2(Phi)))**2
    center = np.array(spectrum.shape) // 2
    spectrum[center[0], center[1]] = 0  # Remove DC component
    peak_idx = np.unravel_index(np.argmax(spectrum), spectrum.shape)
    kx = peak_idx[1] - center[1]
    ky = peak_idx[0] - center[0]
    k_dom = np.sqrt(kx**2 + ky**2) / L
    lambda_dom = 1 / k_dom if k_dom != 0 else np.inf
    return lambda_dom, spectrum

# --- Sweep Execution ---
results = []
if save_images:
    os.makedirs("stability005_outputs", exist_ok=True)

print("Running Stability005 sweep...\n")
for alpha in alpha_values:
    for gamma in gamma_values:
        np.random.seed(seed)
        Phi = np.random.randn(L, L)
        for t in range(T_max):
            Phi += alpha * laplacian(Phi) - beta * Phi + gamma * np.tanh(Phi) + delta * np.random.randn(L, L)
        lambda_dom, spectrum = extract_lambda_dom(Phi)
        results.append((alpha, gamma, lambda_dom))
        print(f"α = {alpha:.2f}, γ = {gamma:.2f} → λ_dom ≈ {lambda_dom:.2f}")

        # --- Visualization ---
        if save_images:
            fig, axs = plt.subplots(1, 2, figsize=(10, 4))
            axs[0].imshow(Phi, cmap='viridis')
            axs[0].set_title(f'Final Field\nα={alpha}, γ={gamma}')
            axs[1].imshow(np.log1p(spectrum), cmap='inferno')
            axs[1].set_title('Power Spectrum (log scale)')
            plt.tight_layout()
            plt.savefig(f"stability005_outputs/stability005_alpha{alpha}_gamma{gamma}.png")
            plt.close()

# --- Summary Table ---
df = pd.DataFrame(results, columns=["alpha", "gamma", "lambda_dom"])
print("\nSweep Complete. Summary Table:\n")
print(df.to_string(index=False))


Running Stability005 sweep...

α = 0.10, γ = 0.02 → λ_dom ≈ 70.71
α = 0.10, γ = 0.05 → λ_dom ≈ 70.71
α = 0.10, γ = 0.10 → λ_dom ≈ 70.71
α = 0.10, γ = 0.15 → λ_dom ≈ 70.71
α = 0.10, γ = 0.20 → λ_dom ≈ 70.71
α = 0.30, γ = 0.02 → λ_dom ≈ 1.43
α = 0.30, γ = 0.05 → λ_dom ≈ 1.43
α = 0.30, γ = 0.10 → λ_dom ≈ 1.43
α = 0.30, γ = 0.15 → λ_dom ≈ 1.43
α = 0.30, γ = 0.20 → λ_dom ≈ 1.43
α = 0.50, γ = 0.02 → λ_dom ≈ 1.43
α = 0.50, γ = 0.05 → λ_dom ≈ 1.43
α = 0.50, γ = 0.10 → λ_dom ≈ 1.43
α = 0.50, γ = 0.15 → λ_dom ≈ 1.43
α = 0.50, γ = 0.20 → λ_dom ≈ 1.43
α = 0.70, γ = 0.02 → λ_dom ≈ 1.43
α = 0.70, γ = 0.05 → λ_dom ≈ 1.43
α = 0.70, γ = 0.10 → λ_dom ≈ 1.43
α = 0.70, γ = 0.15 → λ_dom ≈ 1.43
α = 0.70, γ = 0.20 → λ_dom ≈ 1.43
α = 1.00, γ = 0.02 → λ_dom ≈ 1.43
α = 1.00, γ = 0.05 → λ_dom ≈ 1.43
α = 1.00, γ = 0.10 → λ_dom ≈ 1.43
α = 1.00, γ = 0.15 → λ_dom ≈ 1.43
α = 1.00, γ = 0.20 → λ_dom ≈ 1.43

Sweep Complete. Summary Table:

 alpha  gamma  lambda_dom
   0.1   0.02   70.710678
   0.1   0.05   70.710678
   